# 热轧板带数据预处理

流程：按时间与炉号划分 → 分开有标签/无标签样本 → 特征清洗 → 有标签训练集 KNN 裁剪 → 输出建模数据。

仅有标签训练数据参与特征筛选和中位数拟合。目标列不填充；测试时段无标签数据不并入训练。
原始分区、严重缺失样本和 KNN 删除样本均单独保留，可核对每条样本的去向。

首次使用请重启内核后全部运行。后续通常只需修改“参数与数据”单元格，再运行其后的单元格。
函数定义已内嵌，本 Notebook 可以单独移动使用，无需配套 `.py` 文件。
RTT为常温拉伸下的屈服强度YS


## 1. 函数定义
将参数、时间划分、特征处理、KNN 和结果校验分别封装。


In [ ]:
"""热轧板带数据预处理：完整保留无标签样本，所有拟合只使用有标签训练数据。"""

from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler


EXCLUDE_COLS = (
    "PLATE_NO", "SMP_NO", "RETEST_FLAG", "SLAB_NO", "SLAB_STATUS_FLAG",
    "SLAB_STLGRD", "MILL_STDSPEC", "APLY_STDSPEC", "HEAT_NO",
    "CENTER_SEGREGATION_RSLT", "CENTER_POROSITY_RSLT", "MIDDLE_CRACKS_RSLT",
    "CORNER_CRACKS_RSLT", "TRAINGULAR_CRACKS_RSLT", "AL2O3_INCLUSION_RSLT",
    "PINHOLE_BUBBLE_RSLT", "HONEYCOMB_BUBBLE_RSLT", "MILL_OCCR_DATE",
    "STDSPEC_STAMP", "OCCR_DATE",
)


@dataclass(frozen=True)
class PreprocessConfig:
    """集中管理参数；默认值沿用修正版 Notebook。"""

    target_col: str = "RTT_RPDOT2"
    date_col: str = "MILL_OCCR_DATE"
    heat_col: str = "HEAT_NO"
    cutoff_date: str = "2026-03-16"
    feature_missing_threshold: float = 0.50
    row_missing_threshold: float = 0.40
    k: int = 5
    dense_ratio: float = 0.10
    dense_keep_ratio: float = 0.50
    random_state: int = 42
    exclude_cols: tuple[str, ...] = EXCLUDE_COLS

    def __post_init__(self):
        for name in ("feature_missing_threshold", "row_missing_threshold",
                     "dense_ratio", "dense_keep_ratio"):
            if not 0 <= getattr(self, name) <= 1:
                raise ValueError(f"{name} 必须在 0 到 1 之间。")
        if not isinstance(self.k, int) or isinstance(self.k, bool) or self.k < 1:
            raise ValueError("k 必须为正整数。")
        if pd.isna(pd.Timestamp(self.cutoff_date)):
            raise ValueError("cutoff_date 必须是有效日期。")


@dataclass
class TimePartitions:
    train: pd.DataFrame
    test: pd.DataFrame
    purge: pd.DataFrame
    invalid: pd.DataFrame
    heat_time: pd.DataFrame


@dataclass
class FeaturePreprocessor:
    feature_cols: list[str]
    missing_indicator_cols: list[str]
    drop_feature_cols: list[str]
    train_missing_rate: pd.Series
    imputer: SimpleImputer

    @property
    def indicator_cols(self):
        return [col + "_MISSING" for col in self.missing_indicator_cols]

    @property
    def model_feature_cols(self):
        return self.feature_cols + self.indicator_cols

    @property
    def fill_values(self):
        return pd.Series(self.imputer.statistics_, index=self.feature_cols,
                         name="median_fill_value")

    def transform(self, frame):
        """仅转换输入特征；保留目标缺失状态和样本索引。"""
        result = frame.copy()
        indicators = result[self.missing_indicator_cols].isna().astype("int8")
        indicators.columns = self.indicator_cols
        if len(result):  # 空无标签分区也可以正常输出
            result[self.feature_cols] = self.imputer.transform(result[self.feature_cols])
        return pd.concat([result, indicators], axis=1).copy()


@dataclass
class PreprocessResult:
    config: PreprocessConfig
    periods: TimePartitions
    raw: dict[str, pd.DataFrame]
    cleaned: dict[str, pd.DataFrame]
    rejected: dict[str, pd.DataFrame]
    preprocessor: FeaturePreprocessor
    train_labeled_pre_knn: pd.DataFrame
    knn_removed: pd.DataFrame
    knn_diagnostics: pd.DataFrame

    def features(self, group):
        return self.cleaned[group][self.preprocessor.model_feature_cols].copy()

    def labels(self, group):
        return self.cleaned[group][self.config.target_col].copy()

    def period_frame(self, period):
        return pd.concat([self.cleaned[f"{period}_labeled"],
                          self.cleaned[f"{period}_unlabeled"]]).sort_values(
                              self.config.date_col).copy()

    def summary(self):
        rows = []
        for period, name in (("train", "训练"), ("test", "测试")):
            labeled, unlabeled = f"{period}_labeled", f"{period}_unlabeled"
            before = (self.train_labeled_pre_knn if period == "train"
                      else self.cleaned[labeled])
            rows.append({
                "时段": name,
                "原始总数": len(self.raw[labeled]) + len(self.raw[unlabeled]),
                "原始有标签": len(self.raw[labeled]),
                "原始无标签": len(self.raw[unlabeled]),
                "清洗后有标签（KNN前）": len(before),
                "清洗后无标签": len(self.cleaned[unlabeled]),
                "严重缺失隔离的无标签": len(self.rejected[unlabeled]),
                "最终有标签": len(self.cleaned[labeled]),
            })
        return pd.DataFrame(rows).set_index("时段")


def split_by_time_and_heat(data, config):
    """跨越时间切点的整炉样本隔离，不让同一炉进入训练和测试。"""
    required = {config.date_col, config.heat_col, config.target_col}
    missing = required.difference(data.columns)
    if missing:
        raise ValueError(f"缺少必要字段：{sorted(missing)}")
    if not data.columns.is_unique:
        raise ValueError("输入数据包含重名字段。")
    frame = data.copy(deep=True)
    frame[config.date_col] = pd.to_datetime(frame[config.date_col], errors="coerce")
    invalid_mask = frame[[config.date_col, config.heat_col]].isna().any(axis=1)
    invalid = frame.loc[invalid_mask].copy()
    frame = frame.loc[~invalid_mask].copy()
    heat_time = frame.groupby(config.heat_col)[config.date_col].agg(
        min_date="min", max_date="max", count="count")
    cutoff = pd.Timestamp(config.cutoff_date)
    train_mask = heat_time["max_date"] < cutoff
    test_mask = heat_time["min_date"] >= cutoff

    def select(mask):
        heats = heat_time.index[mask]
        return frame.loc[frame[config.heat_col].isin(heats)].sort_values(
            config.date_col).reset_index(drop=True)

    return TimePartitions(select(train_mask), select(test_mask),
                          select(~train_mask & ~test_mask), invalid, heat_time)


def split_labels(periods, target_col):
    """原始无标签样本保存在独立分区，后续清洗不覆盖它们。"""
    groups = {}
    for name, frame in (("train", periods.train), ("test", periods.test)):
        labeled = frame[target_col].notna()
        groups[f"{name}_labeled"] = frame.loc[labeled].copy()
        groups[f"{name}_unlabeled"] = frame.loc[~labeled].copy()
    return groups


def clean_features(raw, config):
    """学习特征筛选、缺失标记和中位数后，统一转换四类分区。"""
    train = raw["train_labeled"]
    if train.empty:
        raise ValueError("训练时段没有有标签样本，无法拟合预处理。")
    excluded = set(config.exclude_cols) | {config.target_col, config.date_col, config.heat_col}
    features = [col for col in train.select_dtypes(include=[np.number]).columns
                if col not in excluded]
    missing_rate = train[features].isna().mean().sort_values(ascending=False)
    dropped = missing_rate[missing_rate > config.feature_missing_threshold].index.tolist()
    features = [col for col in features if col not in dropped]
    if not features:
        raise ValueError("缺失率筛选后没有可用数值特征。")

    kept, rejected = {}, {}
    for name, frame in raw.items():
        keep = frame[features].isna().mean(axis=1) <= config.row_missing_threshold
        kept[name], rejected[name] = frame.loc[keep].copy(), frame.loc[~keep].copy()
    train = kept["train_labeled"]
    if train.empty:
        raise ValueError("特征缺失率过滤后没有有标签训练样本。")
    all_missing = [col for col in features if train[col].isna().all()]
    features = [col for col in features if col not in all_missing]
    if not features:
        raise ValueError("样本过滤后所有训练特征均为空。")
    indicators = [col for col in features if train[col].isna().any()]
    if set(col + "_MISSING" for col in indicators).intersection(train.columns):
        raise ValueError("缺失标记字段与原字段重名，请先调整字段名。")
    imputer = SimpleImputer(strategy="median").fit(train[features])
    preprocessor = FeaturePreprocessor(features, indicators, dropped + all_missing,
                                      missing_rate, imputer)
    cleaned = {name: preprocessor.transform(frame) for name, frame in kept.items()}
    return cleaned, rejected, preprocessor


def trim_knn(train, feature_cols, config):
    """只裁剪传入的有标签训练数据，不修改输入表。"""
    scaled = StandardScaler().fit_transform(train[feature_cols])
    model = NearestNeighbors(n_neighbors=min(config.k + 1, len(train)),
                             metric="euclidean", algorithm="auto").fit(scaled)
    distances, _ = model.kneighbors(scaled)
    mean_distance = (distances[:, 1:].mean(axis=1) if distances.shape[1] > 1
                     else np.zeros(len(train)))
    diagnostics = pd.DataFrame({"mean_neighbor_distance": mean_distance}, index=train.index)
    threshold = diagnostics["mean_neighbor_distance"].quantile(config.dense_ratio)
    diagnostics["dense_sample"] = diagnostics["mean_neighbor_distance"] <= threshold
    dense = train.loc[diagnostics["dense_sample"]]
    normal = train.loc[~diagnostics["dense_sample"]]
    keep_num = int(round(len(dense) * config.dense_keep_ratio))
    # 沿用原算法：四舍五入为 0 时保留该候选组，避免小样本组被清空。
    dense_kept = (dense.sample(n=keep_num, random_state=config.random_state)
                  if keep_num > 0 else dense)
    trimmed = pd.concat([normal, dense_kept]).sort_values(config.date_col).copy()
    removed = train.loc[~train.index.isin(trimmed.index)].copy()
    return trimmed, removed, diagnostics


def validate_result(result):
    """核对标签、特征、数据去向，以及时间和炉号隔离。"""
    target = result.config.target_col
    for name, frame in result.cleaned.items():
        assert not frame[result.preprocessor.model_feature_cols].isna().any().any(), name
        if name.endswith("_unlabeled"):
            assert frame[target].isna().all(), name
        else:
            assert frame[target].notna().all(), name
        accounted = len(frame) + len(result.rejected[name])
        if name == "train_labeled":
            accounted += len(result.knn_removed)
        assert len(result.raw[name]) == accounted, f"{name} 的样本数量不守恒"
    train, test = result.periods.train, result.periods.test
    assert set(train[result.config.heat_col]).isdisjoint(test[result.config.heat_col])
    if not train.empty and not test.empty:
        assert train[result.config.date_col].max() < test[result.config.date_col].min()


def run_preprocessing(data, config=None):
    """从原始数据运行完整流程；重复调用不会累积删除或修改输入。"""
    config = config if config is not None else PreprocessConfig()
    periods = split_by_time_and_heat(data, config)
    raw = split_labels(periods, config.target_col)
    cleaned, rejected, preprocessor = clean_features(raw, config)
    before = cleaned["train_labeled"].copy(deep=True)
    cleaned["train_labeled"], removed, diagnostics = trim_knn(
        before, preprocessor.feature_cols, config)
    result = PreprocessResult(config, periods, raw, cleaned, rejected,
                              preprocessor, before, removed, diagnostics)
    validate_result(result)
    assert len(data) == sum(len(frame) for frame in
                            (periods.train, periods.test, periods.purge, periods.invalid))
    return result


## 2. 参数与数据
数据只读取一次。默认特征排除列表沿用修正版，可通过 `exclude_cols` 调整。


In [ ]:
# 相对路径基于内核工作目录；请准备自己的 CSV 数据。
file_path = "data/input.csv"
config = PreprocessConfig(
    target_col="RTT_RPDOT2",
    cutoff_date="2026-03-16",
    feature_missing_threshold=0.50,
    row_missing_threshold=0.40,
    k=5,
    dense_ratio=0.10,
    dense_keep_ratio=0.50,
    random_state=42,
)
df = pd.read_csv(file_path)
print("原始样本数：", len(df))


## 3. 运行预处理
每次都从原始 `df` 开始，重复运行不会累积裁剪。


In [ ]:
result = run_preprocessing(df, config)
label_summary = result.summary()
display(label_summary)
print("跨界炉号隔离样本：", len(result.periods.purge))
print("日期或炉号无效样本：", len(result.periods.invalid))
print("KNN 删除有标签训练样本：", len(result.knn_removed))


## 4. 建模数据与追溯数据

- `X_train / y_train`：KNN 裁剪后的有标签训练数据。
- `X_test / y_test`：有标签测试数据，供评估使用。
- `X_train_unlabeled`：训练时段可用无标签特征。
- `X_test_unlabeled`：测试时段可用无标签特征，供未来预测。
- `train_df / test_df`：各时段清洗后完整数据，包含目标为 NaN 的无标签行。
- `result.raw / result.rejected / result.knn_removed`：原始分组、严重缺失隔离和 KNN 删除数据。

字典分组名为 `train_labeled`、`train_unlabeled`、`test_labeled`、`test_unlabeled`。


In [ ]:
# 保留常用变量名，方便接续原有建模代码。
target_col = config.target_col
feature_cols = result.preprocessor.feature_cols.copy()
indicator_cols = result.preprocessor.indicator_cols
model_feature_cols = result.preprocessor.model_feature_cols
imputer = result.preprocessor.imputer
fill_values = result.preprocessor.fill_values

X_train = result.features("train_labeled")
y_train = result.labels("train_labeled")
X_test = result.features("test_labeled")
y_test = result.labels("test_labeled")
X_train_unlabeled = result.features("train_unlabeled")
X_test_unlabeled = result.features("test_unlabeled")

train_period_raw = result.periods.train.copy()
test_period_raw = result.periods.test.copy()
purge_df = result.periods.purge.copy()
train_unlabeled_raw = result.raw["train_unlabeled"].copy()
test_unlabeled_raw = result.raw["test_unlabeled"].copy()
train_labeled_df = result.cleaned["train_labeled"].copy()
test_labeled_df = result.cleaned["test_labeled"].copy()
train_unlabeled_df = result.cleaned["train_unlabeled"].copy()
test_unlabeled_df = result.cleaned["test_unlabeled"].copy()
train_unlabeled_rejected = result.rejected["train_unlabeled"].copy()
test_unlabeled_rejected = result.rejected["test_unlabeled"].copy()
train_df = result.period_frame("train")
test_df = result.period_frame("test")

display(pd.DataFrame({
    "数据": ["有标签训练", "有标签测试", "无标签训练", "无标签测试"],
    "样本数": [len(X_train), len(X_test), len(X_train_unlabeled), len(X_test_unlabeled)],
    "特征数": [len(model_feature_cols)] * 4,
}).set_index("数据"))


## 5. 处理诊断
检查炉次跨度、被删除的特征、中位数和 KNN 邻域距离。


In [ ]:
heat_span = result.periods.heat_time.copy()
heat_span["跨度天数"] = (
    heat_span["max_date"] - heat_span["min_date"]
).dt.total_seconds() / 86400
display(heat_span.sort_values("跨度天数", ascending=False).head(30))
print("删除的特征：", result.preprocessor.drop_feature_cols)
display(fill_values.head(20))
display(result.knn_diagnostics["mean_neighbor_distance"].describe())


## 6. 回归半监督学习：筛选伪标签并扩充训练数据

`RTT_RPDOT2` 是连续回归目标。本节使用随机森林回归自训练：

1. 训练时段内部再按时间和炉号划分验证集；内部预处理也只在内部训练集拟合。
2. 用真实标签训练教师模型，为内部训练时段的无标签样本预测；按树间预测标准差筛选，每轮最多加入原无标签池的 25%。
3. 真实标签权重为 1，伪标签权重为 0.3。最多迭代 3 轮，用内部验证 MAE 选择半监督候选轮数。
4. 按选定轮数在完整训练时段重新训练。测试时段的有标签/无标签样本都不参与这一步。
5. 若内部验证没有改善，仍保留半监督候选及增强数据供研究，推荐模型回退到纯监督基线。

伪标签是模型估计值，不能当作实测标签。树间标准差是启发式筛选分数，不是校准置信区间；不满足阈值的样本继续保留为无标签。
特征缺失严重的隔离样本不参与伪标签训练。沿用上一版特征清单，实际预测时需确保这些特征均可获得。

参考：[RandomForestRegressor 与 sample_weight](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)。
官方 [SelfTrainingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.semi_supervised.SelfTrainingClassifier.html) 面向分类，因此此处单独实现回归自训练。


In [ ]:
"""回归自训练：筛选训练时段伪标签，按时间/炉号内部验证，测试集仅评估。"""

from dataclasses import dataclass, replace

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



@dataclass(frozen=True)
class SemiSupervisedConfig:
    validation_fraction: float = 0.20
    max_iter: int = 3
    batch_fraction: float = 0.25
    pseudo_weight: float = 0.30
    uncertainty_ratio: float = 0.35
    n_estimators: int = 200
    min_samples_leaf: int = 3
    max_features: float = 0.70
    random_state: int = 42
    n_jobs: int = -1

    def __post_init__(self):
        if not 0 < self.validation_fraction < 1:
            raise ValueError("validation_fraction 必须在 0 和 1 之间。")
        for name in ("batch_fraction", "pseudo_weight", "max_features"):
            if not 0 < getattr(self, name) <= 1:
                raise ValueError(f"{name} 必须在 (0, 1] 内。")
        if not np.isfinite(self.uncertainty_ratio) or self.uncertainty_ratio <= 0:
            raise ValueError("uncertainty_ratio 必须是有限正数。")
        for name in ("max_iter", "n_estimators", "min_samples_leaf"):
            value = getattr(self, name)
            if not isinstance(value, int) or isinstance(value, bool) or value < 1:
                raise ValueError(f"{name} 必须为正整数。")


def regression_metrics(y_true, prediction):
    if len(y_true) == 0:
        return {"MAE": np.nan, "RMSE": np.nan, "R2": np.nan}
    return {
        "MAE": mean_absolute_error(y_true, prediction),
        "RMSE": np.sqrt(mean_squared_error(y_true, prediction)),
        "R2": r2_score(y_true, prediction) if len(y_true) > 1 else np.nan,
    }


def make_regressor(config):
    return RandomForestRegressor(
        n_estimators=config.n_estimators,
        min_samples_leaf=config.min_samples_leaf,
        max_features=config.max_features,
        random_state=config.random_state,
        n_jobs=config.n_jobs,
    )


def predict_with_disagreement(model, features):
    """树间标准差仅作为筛选分数，不是经过校准的置信区间。"""
    if features.empty:
        return np.empty(0), np.empty(0)
    tree_predictions = np.vstack([
        tree.predict(features.to_numpy(dtype=float)) for tree in model.estimators_
    ])
    return tree_predictions.mean(axis=0), tree_predictions.std(axis=0)


@dataclass
class SelfTrainingRun:
    baseline: RandomForestRegressor
    model: RandomForestRegressor
    X_augmented: pd.DataFrame
    y_augmented: pd.Series
    sample_weight: pd.Series
    pseudo_labels: pd.DataFrame
    history: pd.DataFrame


def self_train_regressor(X_labeled, y_labeled, X_unlabeled, config,
                         X_validation=None, y_validation=None, rounds=None):
    """从真实标签开始逐轮加入低分歧预测；不改写任何输入。"""
    if not X_labeled.index.equals(y_labeled.index):
        raise ValueError("有标签特征与标签索引不一致。")
    if not X_labeled.columns.equals(X_unlabeled.columns):
        raise ValueError("有标签与无标签特征列不一致。")
    if (not X_labeled.index.is_unique or not X_unlabeled.index.is_unique
            or len(X_labeled.index.intersection(X_unlabeled.index))):
        raise ValueError("同一训练分区内的样本索引必须唯一且互不重叠。")
    if len(X_labeled) < 2 or not np.isfinite(y_labeled.to_numpy(dtype=float)).all():
        raise ValueError("至少需要两条具有有限真实标签的训练样本。")
    if (X_validation is None) != (y_validation is None):
        raise ValueError("验证特征和标签必须同时提供。")
    if X_validation is not None:
        if (not X_validation.columns.equals(X_labeled.columns)
                or not X_validation.index.equals(y_validation.index)
                or X_validation.empty):
            raise ValueError("验证数据必须非空，且特征列和标签索引需对齐。")
    for frame in [X_labeled, X_unlabeled] + ([] if X_validation is None else [X_validation]):
        if not np.isfinite(frame.to_numpy(dtype=float)).all():
            raise ValueError("输入特征包含缺失或无穷值，请先完成预处理。")
    rounds = config.max_iter if rounds is None else rounds
    if not isinstance(rounds, int) or not 0 <= rounds <= config.max_iter:
        raise ValueError("rounds 必须在 0 到 max_iter 之间。")

    X_augmented = X_labeled.copy()
    y_augmented = y_labeled.astype(float).copy()
    weights = pd.Series(1.0, index=X_labeled.index, name="sample_weight")
    pseudo = pd.DataFrame(index=X_unlabeled.index)
    pseudo["pseudo_target"] = np.nan
    pseudo["tree_std"] = np.nan
    pseudo["iteration"] = 0
    pseudo["selected"] = False
    # 阈值只由真实训练标签的尺度决定；它是启发式超参数。
    threshold = config.uncertainty_ratio * float(y_labeled.std(ddof=0))
    batch_size = max(1, int(np.ceil(len(X_unlabeled) * config.batch_fraction)))
    baseline = make_regressor(config).fit(X_augmented, y_augmented, sample_weight=weights)
    model = baseline
    history = []

    def record(iteration, added, reason):
        row = {"iteration": iteration, "added": added,
               "pseudo_total": int(pseudo["selected"].sum()),
               "std_threshold": threshold, "status": reason}
        if X_validation is not None:
            row.update(regression_metrics(y_validation, model.predict(X_validation)))
        history.append(row)

    record(0, 0, "supervised_baseline")
    for iteration in range(1, rounds + 1):
        remaining = X_unlabeled.loc[~pseudo["selected"]]
        if remaining.empty:
            break
        prediction, disagreement = predict_with_disagreement(model, remaining)
        candidates = pd.DataFrame({"prediction": prediction, "std": disagreement},
                                  index=remaining.index)
        eligible = candidates.loc[candidates["std"] <= threshold]
        selected = eligible.sort_values("std", kind="stable").head(batch_size)
        if selected.empty:
            record(iteration, 0, "no_eligible_pseudo_labels")
            break
        ids = selected.index
        pseudo.loc[ids, "pseudo_target"] = selected["prediction"]
        pseudo.loc[ids, "tree_std"] = selected["std"]
        pseudo.loc[ids, "iteration"] = iteration
        pseudo.loc[ids, "selected"] = True
        X_augmented = pd.concat([X_augmented, X_unlabeled.loc[ids]])
        new_y = selected["prediction"].rename(y_labeled.name)
        y_augmented = pd.concat([y_augmented, new_y])
        weights = pd.concat([weights, pd.Series(config.pseudo_weight, index=ids,
                                               name="sample_weight")])
        model = make_regressor(config).fit(X_augmented, y_augmented, sample_weight=weights)
        record(iteration, len(ids), "pseudo_labels_added")

    # 保存所有无标签样本的最终预测，但只有 selected=True 的行进入增强训练集。
    pseudo["final_prediction"], pseudo["final_tree_std"] = predict_with_disagreement(
        model, X_unlabeled)
    pseudo["sample_weight"] = np.where(pseudo["selected"], config.pseudo_weight, 0.0)
    pd.testing.assert_series_equal(y_augmented.loc[y_labeled.index],
                                   y_labeled.astype(float))
    assert X_augmented.index.equals(y_augmented.index) and weights.index.equals(y_augmented.index)
    assert len(X_augmented) == len(X_labeled) + int(pseudo["selected"].sum())
    return SelfTrainingRun(baseline, model, X_augmented, y_augmented,
                           weights, pseudo, pd.DataFrame(history))


@dataclass
class SemiSupervisedResult:
    config: SemiSupervisedConfig
    validation_cutoff: pd.Timestamp
    validation_history: pd.DataFrame
    validation_partition_summary: pd.DataFrame
    selected_rounds: int
    improves_validation: bool
    final_run: SelfTrainingRun
    train_augmented: pd.DataFrame
    unlabeled_predictions: pd.DataFrame

    @property
    def recommended_model(self):
        return self.final_run.model if self.improves_validation else self.final_run.baseline


def fit_semi_supervised(preprocessed, config=None):
    """测试数据不参与选轮数、筛伪标签或训练；内部验证重新拟合预处理。"""
    config = config or SemiSupervisedConfig()
    pc = preprocessed.config
    train_raw = preprocessed.periods.train.copy(deep=True)
    heat_end = train_raw.groupby(pc.heat_col)[pc.date_col].max().sort_values()
    if len(heat_end) < 5:
        raise ValueError("训练时段至少需要 5 个炉号，以进行按时间/炉号的内部验证。")
    position = min(len(heat_end) - 1, max(1, int(len(heat_end) * (1 - config.validation_fraction))))
    cutoff = pd.Timestamp(heat_end.iloc[position])
    # 先切分，再用内部训练集拟合特征筛选/填充/KNN，避免验证信息泄漏。
    inner = run_preprocessing(train_raw, replace(pc, cutoff_date=cutoff))
    X_validation, y_validation = inner.features("test_labeled"), inner.labels("test_labeled")
    if X_validation.empty:
        raise ValueError("内部验证集没有可用真实标签，请调整 validation_fraction。")
    trial = self_train_regressor(
        inner.features("train_labeled"), inner.labels("train_labeled"),
        inner.features("train_unlabeled"), config, X_validation, y_validation,
    )
    baseline_mae = float(trial.history.iloc[0]["MAE"])
    candidate_rounds = trial.history.loc[trial.history["added"] > 0]
    if candidate_rounds.empty:
        rounds, improved = 0, False
    else:
        best = candidate_rounds.sort_values(["MAE", "iteration"]).iloc[0]
        rounds = int(best["iteration"])
        improved = bool(best["MAE"] < baseline_mae)

    # 即使内部验证未改善，也保留最佳半监督候选供比较；recommended_model 回退基线。
    final_run = self_train_regressor(
        preprocessed.features("train_labeled"), preprocessed.labels("train_labeled"),
        preprocessed.features("train_unlabeled"), config, rounds=rounds,
    )
    audit = preprocessed.cleaned["train_unlabeled"].copy()
    audit = audit.join(final_run.pseudo_labels)
    assert audit[pc.target_col].isna().all()
    selected = audit.loc[audit["selected"]].copy()
    selected["_ORIGINAL_TARGET"] = selected[pc.target_col]
    selected[pc.target_col] = selected["pseudo_target"]
    selected["_LABEL_SOURCE"] = "pseudo"
    selected["_SAMPLE_WEIGHT"] = config.pseudo_weight
    selected["_PSEUDO_ITER"] = selected["iteration"]
    selected["_PSEUDO_STD"] = selected["tree_std"]
    # 审计字段不作为输入特征；始终使用 preprocessor.model_feature_cols。
    selected = selected[preprocessed.cleaned["train_unlabeled"].columns.tolist() +
                        ["_ORIGINAL_TARGET", "_LABEL_SOURCE", "_SAMPLE_WEIGHT",
                         "_PSEUDO_ITER", "_PSEUDO_STD"]]
    real = preprocessed.cleaned["train_labeled"].copy()
    real["_ORIGINAL_TARGET"] = real[pc.target_col]
    real["_LABEL_SOURCE"] = "measured"
    real["_SAMPLE_WEIGHT"] = 1.0
    real["_PSEUDO_ITER"] = 0
    real["_PSEUDO_STD"] = np.nan
    augmented = pd.concat([real, selected]).loc[final_run.X_augmented.index].copy()
    pd.testing.assert_series_equal(augmented[pc.target_col].astype(float),
                                   final_run.y_augmented)
    return SemiSupervisedResult(config, cutoff, trial.history, inner.summary(),
                                rounds, improved, final_run, augmented, audit)


def evaluate_semi_supervised(semi_result, X_test, y_test):
    """训练完成后调用；这里只报告测试指标，不修改模型选择。"""
    if not X_test.index.equals(y_test.index):
        raise ValueError("测试特征与标签索引不一致。")
    rows = []
    for name, model in (("纯监督基线", semi_result.final_run.baseline),
                        ("半监督候选", semi_result.final_run.model)):
        prediction = model.predict(X_test) if len(X_test) else np.empty(0)
        rows.append({"模型": name, **regression_metrics(y_test, prediction)})
    return pd.DataFrame(rows).set_index("模型")


## 7. 训练与内部验证
阈值为 `uncertainty_ratio × 真实训练标签标准差`，不强制给全部无标签样本补值。


In [ ]:
semi_config = SemiSupervisedConfig(
    validation_fraction=0.20,
    max_iter=3,
    batch_fraction=0.25,
    pseudo_weight=0.30,
    uncertainty_ratio=0.35,
    n_estimators=200,
    random_state=42,
)
semi_result = fit_semi_supervised(result, semi_config)
print("内部验证切分时间：", semi_result.validation_cutoff)
display(semi_result.validation_partition_summary)
display(semi_result.validation_history)
print("选用的半监督迭代轮数：", semi_result.selected_rounds)
print("内部验证 MAE 是否优于纯监督：", semi_result.improves_validation)


## 8. 获取补充标签后的训练集

`train_augmented_df` 中的目标列包含真实标签和筛选后的伪标签，`_LABEL_SOURCE` 区分来源。
`_ORIGINAL_TARGET` 保存原值；原始数据与 `train_unlabeled_raw` 不被修改。
`pseudo_label_audit` 保留全部可用训练无标签样本的预测和是否入选；未入选行的目标列仍为空。

后续训练使用 `X_train_semi`、`y_train_semi` 和 `sample_weight_semi`，不要把来源、权重或伪标签审计字段作为输入特征。


In [ ]:
X_train_semi = semi_result.final_run.X_augmented.copy()
y_train_semi = semi_result.final_run.y_augmented.copy()
sample_weight_semi = semi_result.final_run.sample_weight.copy()
train_augmented_df = semi_result.train_augmented.copy()
pseudo_label_audit = semi_result.unlabeled_predictions.copy()
remaining_unlabeled_df = train_unlabeled_df.loc[
    ~pseudo_label_audit["selected"]
].copy()
semi_model = semi_result.final_run.model
baseline_model = semi_result.final_run.baseline
recommended_model = semi_result.recommended_model

print("真实有标签训练样本：", len(X_train))
print("清洗后训练无标签样本：", len(X_train_unlabeled))
print("已筛选伪标签：", int(pseudo_label_audit["selected"].sum()))
print("增强训练样本：", len(X_train_semi))
print("仍未采用的无标签样本：", len(remaining_unlabeled_df))
print("特征严重缺失的原始无标签样本：", len(train_unlabeled_rejected))
display(semi_result.final_run.history)
display(train_augmented_df[[target_col, "_ORIGINAL_TARGET", "_LABEL_SOURCE",
                            "_SAMPLE_WEIGHT", "_PSEUDO_ITER", "_PSEUDO_STD"]].tail(10))

# 示例：训练另一个支持 sample_weight 的回归器时，同样传入权重。
# your_model.fit(X_train_semi, y_train_semi, sample_weight=sample_weight_semi)
assert train_unlabeled_raw[target_col].isna().all()
assert train_unlabeled_df[target_col].isna().all()
assert y_train_semi.notna().all()


## 9. 最终测试评估
只在训练和内部选轮数完成后报告指标；不使用测试结果调整伪标签阈值或选择轮数。


In [ ]:
test_metrics = evaluate_semi_supervised(semi_result, X_test, y_test)
display(test_metrics)
print("按内部验证推荐：", "半监督模型" if semi_result.improves_validation else "纯监督基线")


## 10. 导出增强训练集与伪标签审计表

保存到当前 Notebook 工作目录下的 `semi_supervised_outputs`。重复运行会更新该目录内的结果文件，不修改原始 CSV。


In [ ]:
from pathlib import Path

export_dir = Path("semi_supervised_outputs")
export_dir.mkdir(parents=True, exist_ok=True)
train_augmented_df.to_csv(export_dir / "train_augmented.csv", index_label="train_row_id", encoding="utf-8-sig")
pseudo_label_audit.to_csv(export_dir / "pseudo_label_audit.csv", index_label="train_row_id", encoding="utf-8-sig")
remaining_unlabeled_df.to_csv(export_dir / "remaining_unlabeled.csv", index_label="train_row_id", encoding="utf-8-sig")
semi_result.validation_history.to_csv(export_dir / "validation_history.csv", index=False, encoding="utf-8-sig")
test_metrics.to_csv(export_dir / "test_metrics.csv", encoding="utf-8-sig")
print("结果目录：", export_dir.resolve())
